####  Imports

In [2]:
import pandas as pd
import numpy as np

df_events    = pd.read_csv("../data/shipment_events.csv")
df_suppliers = pd.read_csv("../data/supplier_master.csv")

print(f"Events: {df_events.shape}")
print(f"Suppliers: {df_suppliers.shape}")

Events: (8000, 22)
Suppliers: (120, 9)


#### Supplier scorecard rollup

In [3]:
grp = df_events.groupby("supplier_id")

# Build each KPI as its own series, then combine
total_shipments    = grp.size().rename("total_shipments")
total_violations   = grp["has_violation"].sum().rename("total_violations")
chargebacks_issued = grp["chargeback_issued"].sum().rename("chargebacks_issued")
false_positives    = grp["is_false_positive"].sum().rename("false_positives_suppressed")
total_penalty      = grp["final_penalty_usd"].sum().rename("total_penalty_usd")
disputes_raised    = grp["dispute_raised"].sum().rename("disputes_raised")

# OTIF rate — % of shipments that were on time and in full
otif_col = ~(
    (df_events["violation_type"] == "OTIF_MISS") &
    df_events["chargeback_issued"]
)
df_events["on_time_in_full"] = otif_col
otif_rate = grp["on_time_in_full"].mean().rename("otif_rate")

# ASN accuracy — % of shipments with no ASN error
df_events["asn_ok"] = ~df_events["violation_type"].isin(
    ["ASN_LATE", "ASN_MISSING"]
) | df_events["is_false_positive"]
asn_rate = grp["asn_ok"].mean().rename("asn_accuracy_rate")

# Combine all KPIs into one scorecard
scorecard = pd.concat([
    total_shipments, total_violations, chargebacks_issued,
    false_positives, total_penalty, disputes_raised,
    otif_rate, asn_rate
], axis=1).reset_index()

print(scorecard.head())

  supplier_id  total_shipments  total_violations  chargebacks_issued  \
0    SUPP_001               63                 7                   6   
1    SUPP_002               56                 3                   3   
2    SUPP_003               76                11                  10   
3    SUPP_004               64                 3                   2   
4    SUPP_005               56                 3                   3   

   false_positives_suppressed  total_penalty_usd  disputes_raised  otif_rate  \
0                           1           60837.71                0   1.000000   
1                           0            2619.07                1   1.000000   
2                           1           21954.26                2   0.973684   
3                           1            1688.00                1   1.000000   
4                           0           51078.55                0   1.000000   

   asn_accuracy_rate  
0           0.984127  
1           0.982143  
2           0.986

#### Merge supplier attributes and calculate compliance rate

In [4]:
scorecard = scorecard.merge(
    df_suppliers[[
        "supplier_id", "supplier_name", "tier",
        "part_category", "region", "primary_plant",
        "contract_value_usd", "risk_profile"
    ]],
    on="supplier_id"
)

# Compliance rate = % of shipments with no chargeback
scorecard["compliance_rate"] = round(
    1 - (scorecard["chargebacks_issued"] / scorecard["total_shipments"]), 4
)

# False positive rate = FPs caught / total violations flagged
scorecard["fp_suppression_rate"] = round(
    scorecard["false_positives_suppressed"] /
    scorecard["total_violations"].clip(lower=1), 4
)

scorecard["otif_rate"]         = round(scorecard["otif_rate"], 4)
scorecard["asn_accuracy_rate"] = round(scorecard["asn_accuracy_rate"], 4)
scorecard["total_penalty_usd"] = round(scorecard["total_penalty_usd"], 2)

print(scorecard[["supplier_id","compliance_rate","otif_rate","total_penalty_usd"]].head(10))

  supplier_id  compliance_rate  otif_rate  total_penalty_usd
0    SUPP_001           0.9048     1.0000           60837.71
1    SUPP_002           0.9464     1.0000            2619.07
2    SUPP_003           0.8684     0.9737           21954.26
3    SUPP_004           0.9688     1.0000            1688.00
4    SUPP_005           0.9464     1.0000           51078.55
5    SUPP_006           0.8235     1.0000          142074.28
6    SUPP_007           0.9692     0.9846           22702.80
7    SUPP_008           0.8644     0.9661           34810.01
8    SUPP_009           0.8750     0.9688           34539.36
9    SUPP_010           0.6769     0.9846           96511.25


#### Risk score and risk tier

In [5]:
# Percentile rank each component (0-100, higher = worse)
def pct_rank(series):
    return series.rank(pct=True) * 100

scorecard["risk_score"] = round(
    pct_rank(1 - scorecard["compliance_rate"]) * 0.40 +
    pct_rank(1 - scorecard["otif_rate"])        * 0.30 +
    pct_rank(1 - scorecard["asn_accuracy_rate"])* 0.20 +
    pct_rank(1 - scorecard["fp_suppression_rate"]) * 0.10,
    1
)

scorecard["risk_tier"] = pd.cut(
    scorecard["risk_score"],
    bins=[-1, 25, 50, 75, 101],
    labels=["Green — Low Risk", "Amber — Watch",
            "Orange — At Risk", "Red — Critical"]
)

print("=== RISK TIER DISTRIBUTION ===")
print(scorecard["risk_tier"].value_counts().sort_index())

print("\n=== TOP 5 CRITICAL SUPPLIERS ===")
top5 = scorecard.nlargest(5, "risk_score")[[
    "supplier_id", "region", "risk_score",
    "risk_tier", "compliance_rate", "total_penalty_usd"
]]
print(top5.to_string(index=False))

=== RISK TIER DISTRIBUTION ===
risk_tier
Green — Low Risk    19
Amber — Watch       45
Orange — At Risk    32
Red — Critical      24
Name: count, dtype: int64

=== TOP 5 CRITICAL SUPPLIERS ===
supplier_id  region  risk_score      risk_tier  compliance_rate  total_penalty_usd
   SUPP_117 NA-East        95.0 Red — Critical           0.6757           77648.27
   SUPP_016 NA-East        93.0 Red — Critical           0.7077           74354.88
   SUPP_017 NA-West        92.5 Red — Critical           0.6949           70639.09
   SUPP_113 NA-East        89.8 Red — Critical           0.6620           60995.45
   SUPP_023 NA-West        88.1 Red — Critical           0.6901          103358.61


#### Build the 3 additional export tables

In [6]:
# Monthly trends — for the trend line chart in Tableau
monthly_trends = df_events.groupby(
    ["ship_month", "violation_type"]
).agg(
    shipments   = ("shipment_id", "count"),
    chargebacks = ("chargeback_issued", "sum"),
    penalty_usd = ("final_penalty_usd", "sum"),
    false_positives = ("is_false_positive", "sum")
).reset_index()

# Dispute summary — for the dispute funnel chart
dispute_summary = (
    df_events[df_events["dispute_raised"]]
    .groupby(["region", "violation_type", "dispute_status"])
    .agg(
        count         = ("shipment_id", "count"),
        total_penalty = ("final_penalty_usd", "sum")
    )
    .reset_index()
)

# Regional summary — for the NA vs EMEA heatmap
regional_summary = scorecard.groupby("region").agg(
    suppliers          = ("supplier_id", "count"),
    avg_compliance     = ("compliance_rate", "mean"),
    avg_otif           = ("otif_rate", "mean"),
    total_penalty_usd  = ("total_penalty_usd", "sum"),
    critical_suppliers = ("risk_tier", lambda x: (x == "Red — Critical").sum())
).reset_index()

print("Monthly trends:", monthly_trends.shape)
print("Dispute summary:", dispute_summary.shape)
print("Regional summary:", regional_summary.shape)

Monthly trends: (135, 6)
Dispute summary: (116, 5)
Regional summary: (5, 6)


#### Save all 4 CSVs

In [7]:
import os
os.makedirs("../data", exist_ok=True)

scorecard.to_csv("../data/supplier_scorecards.csv", index=False)
monthly_trends.to_csv("../data/monthly_trends.csv", index=False)
dispute_summary.to_csv("../data/dispute_summary.csv", index=False)
regional_summary.to_csv("../data/regional_summary.csv", index=False)

print("All 4 files saved.")
print(f"  supplier_scorecards : {scorecard.shape}")
print(f"  monthly_trends      : {monthly_trends.shape}")
print(f"  dispute_summary     : {dispute_summary.shape}")
print(f"  regional_summary    : {regional_summary.shape}")

# Final headline numbers
print("\n=== FINAL PROJECT NUMBERS ===")
print(f"Suppliers analyzed       : {len(scorecard)}")
print(f"Shipment events          : {len(df_events):,}")
print(f"Total violations         : {int(df_events['has_violation'].sum()):,}")
print(f"False positives caught   : {int(df_events['is_false_positive'].sum()):,}")
print(f"Chargebacks issued       : {int(df_events['chargeback_issued'].sum()):,}")
print(f"Total penalty recovered  : ${df_events['final_penalty_usd'].sum():,.0f}")
print(f"Disputes tracked         : {int(df_events['dispute_raised'].sum()):,}")
print(f"Critical risk suppliers  : {(scorecard['risk_tier']=='Red — Critical').sum()}")

All 4 files saved.
  supplier_scorecards : (120, 20)
  monthly_trends      : (135, 6)
  dispute_summary     : (116, 5)
  regional_summary    : (5, 6)

=== FINAL PROJECT NUMBERS ===
Suppliers analyzed       : 120
Shipment events          : 8,000
Total violations         : 1,159
False positives caught   : 197
Chargebacks issued       : 962
Total penalty recovered  : $4,733,836
Disputes tracked         : 241
Critical risk suppliers  : 24


In [8]:
print("=" * 50)
print("  TESLA SOS CHARGEBACK INTELLIGENCE SYSTEM")
print("  Final Pipeline Summary")
print("=" * 50)
print(f"  Suppliers analyzed       : {len(scorecard)}")
print(f"  Shipment events          : {len(df_events):,}")
print(f"  Violations detected      : {int(df_events['has_violation'].sum()):,}")
print(f"  False positives caught   : {int(df_events['is_false_positive'].sum()):,}  (17.0% suppressed)")
print(f"  Chargebacks issued       : {int(df_events['chargeback_issued'].sum()):,}")
print(f"  Total penalty recovered  : ${df_events['final_penalty_usd'].sum():,.0f}")
print(f"  Disputes tracked         : {int(df_events['dispute_raised'].sum()):,}")
print(f"  Critical risk suppliers  : {(scorecard['risk_tier']=='Red — Critical').sum()}")
print("=" * 50)

  TESLA SOS CHARGEBACK INTELLIGENCE SYSTEM
  Final Pipeline Summary
  Suppliers analyzed       : 120
  Shipment events          : 8,000
  Violations detected      : 1,159
  False positives caught   : 197  (17.0% suppressed)
  Chargebacks issued       : 962
  Total penalty recovered  : $4,733,836
  Disputes tracked         : 241
  Critical risk suppliers  : 24
